# ML Network Intrusion Detector
**Project by: HEMAA S**

## Project Overview
This project demonstrates a beginner-friendly machine-learning workflow for classifying network traffic as **Benign** or **Potentially Malicious** using a labeled CSV dataset.

> Educational prototype only. It is not a production security system and should not be used as the sole basis for security decisions.

## Dataset
Use the official [CIC-IDS2017 dataset page](https://www.unb.ca/cic/datasets/ids-2017.html). Start with a manageable CSV subset to avoid memory issues.

## How to use
1. Download a labeled CIC-IDS2017 flow CSV file.
2. Place it in the same folder as this notebook and rename it to `network_traffic.csv`.
3. Run the notebook cells in order.
4. Review the evaluation metrics and charts.


## 1. Install / import libraries
If needed, install the packages listed in `requirements.txt` in your Python environment.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
DATA_PATH = Path("network_traffic.csv")
print("Libraries imported successfully.")


## 2. Load the dataset
The notebook expects a CSV containing network-flow features and a label column. CIC-IDS2017 files commonly use a `Label` column, but this cell tries to detect common alternatives.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Download a labeled CSV from the official CIC-IDS2017 page, "
        "place it beside this notebook, and name it network_traffic.csv."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = [str(col).strip() for col in df.columns]

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())


## 3. Clean the data
This step removes duplicate rows, normalizes column names, identifies the label column, and handles invalid numeric values. Make sure the label column really represents benign/attack categories.


In [ ]:
# Find a likely target/label column.
label_candidates = ["Label", "label", "Class", "class", "Target", "target"]
target_col = next((c for c in label_candidates if c in df.columns), None)

if target_col is None:
    raise ValueError(
        "Could not find a label column. Rename the dataset's target column to 'Label' and rerun."
    )

df = df.drop_duplicates().copy()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(str).str.strip()

# Remove rows where the label is empty.
df = df[df[target_col].ne("")].copy()

print("Target column:", target_col)
print("Cleaned dataset shape:", df.shape)
print("\nLabel distribution:")
display(df[target_col].value_counts().to_frame("count"))


## 4. Prepare features and labels
To keep the starter project manageable, this example uses a stratified sample if the CSV is very large. It removes columns that are entirely empty and encodes categorical features safely inside a preprocessing pipeline.


In [ ]:
# Optional cap for beginner laptops. Increase or set to None for a larger dataset.
MAX_ROWS = 100_000

if MAX_ROWS is not None and len(df) > MAX_ROWS:
    # Stratified sampling keeps label proportions approximately similar.
    df, _ = train_test_split(
        df, train_size=MAX_ROWS, random_state=RANDOM_STATE, stratify=df[target_col]
    )

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

# Remove columns with no usable values.
X = X.dropna(axis=1, how="all")

# Convert columns that are numeric but stored as text where possible.
for col in X.columns:
    if X[col].dtype == "object":
        converted = pd.to_numeric(X[col], errors="coerce")
        if converted.notna().mean() > 0.90:
            X[col] = converted

# Drop identifier-like columns if present; they may encourage memorization.
identifier_names = {"flow id", "flow_id", "src ip", "source ip", "dst ip", "destination ip", "timestamp"}
drop_cols = [c for c in X.columns if c.strip().lower() in identifier_names]
if drop_cols:
    X = X.drop(columns=drop_cols)
    print("Dropped identifier-like columns:", drop_cols)

# Use string labels directly; classifiers support string targets.
y = y.astype(str)

print("Rows used:", len(X))
print("Features used:", X.shape[1])
print("Number of classes:", y.nunique())


## 5. Split data and build the model
The split keeps class proportions similar in training and testing. A Random Forest is a strong baseline for tabular data, though performance depends on the dataset and class balance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", max_categories=30))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        random_state=RANDOM_STATE,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

print("Training model...")
model.fit(X_train, y_train)
print("Training complete.")


## 6. Evaluate the model
Accuracy alone may hide poor detection of less common attack classes. Review precision, recall, F1-score, and the confusion matrix as well.


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred)])))
cm = confusion_matrix(y_test, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.2), max(6, len(labels) * 0.9)))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()


## 7. Visualize label distribution
This chart helps show whether the dataset is imbalanced.


In [ ]:
label_counts = y.value_counts()
plt.figure(figsize=(10, 5))
label_counts.plot(kind="bar")
plt.title("Network Traffic Label Distribution")
plt.xlabel("Traffic label")
plt.ylabel("Number of records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 8. Try a sample prediction
This uses an existing test row as an example. It is not a substitute for validating on new, real-world traffic.


In [ ]:
if len(X_test) > 0:
    sample = X_test.iloc[[0]]
    prediction = model.predict(sample)[0]
    print("Predicted label:", prediction)
    print("Actual label:   ", y_test.iloc[0])


## 9. Conclusion
This project demonstrates a baseline ML workflow for network intrusion detection:
- Load and clean labeled traffic data
- Preprocess numeric and categorical features
- Train a Random Forest classifier
- Evaluate predictions using multiple metrics
- Visualize label distribution and classification errors

### Limitations
- Results depend on dataset quality and class balance.
- Public benchmark performance does not guarantee real-world detection.
- This notebook does not capture live network traffic or block attacks.
- Further work could compare models, tune thresholds, and test on a separate dataset.
